# NB-03 — めぐ指数: 斤量補正の検証

**目的**: NB-02 で推定した β₃（斤量補正係数）を理論値 0.2秒/kg と比較し、妥当性を検証する

**理論背景**: 競馬の常識では「1kg 増加 ≒ 0.2秒のペナルティ（2000m基準）」。距離スケールを掛けているため、距離別には:
- 1200m: 0.2 × 0.6 = 0.12 秒/kg
- 2000m: 0.2 × 1.0 = 0.20 秒/kg (基準)
- 3200m: 0.2 × 1.6 = 0.32 秒/kg

**検証観点**:
1. β₃ の点推定値が 0.2 付近にあるか
2. 95%信頼区間が [0.10, 0.40] 程度に収まっているか
3. 距離帯別に β₃ を別推定した場合でも頑健か

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from pathlib import Path
from scipy import stats

INPUT_NB01 = Path('output/nb01')
INPUT_NB02 = Path('output/nb02')
OUTPUT_DIR = Path('output/nb03')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INPUT_NB01 / 'megu_dataset.parquet')
print(f'読み込み完了: {len(df):,} 行')

## 1. NB-02 の β₃ 確認

In [ ]:
from src.db.session import get_session, init_engine
from sqlalchemy import text

init_engine()

with get_session() as session:
    params_df = pd.read_sql(text("""
        SELECT param_name, param_value
        FROM megu_regression_params
        WHERE model_version = 'v1'
    """), session.bind)

params = dict(zip(params_df['param_name'], params_df['param_value']))
beta_weight = float(params.get('beta_weight', 0))

print(f'β₃ (beta_weight) = {beta_weight:.4f} 秒/kg')
print(f'理論値との差: {abs(beta_weight - 0.2):.4f} 秒/kg')
print(f'理論値 (0.2) に対する比率: {beta_weight / 0.2:.2%}')

## 2. 斤量補正の直接観察（ペアマッチング）

In [ ]:
# 同一レース内で斤量が異なる馬のペアを比較
# 斤量差 × 理論係数 が タイム差に反映されているかを確認

STD_WEIGHT_MALE   = 55.0
STD_WEIGHT_FEMALE = 53.0

df['std_weight'] = np.where(df['sex'] == '牝', STD_WEIGHT_FEMALE, STD_WEIGHT_MALE)
df['weight_entry'] = df.get('weight_carried', df.get('weight_entry', np.nan))
df['weight_dev'] = df['weight_entry'] - df['std_weight']
df['dist_scale'] = df['distance'] / 2000.0

# 同一距離×馬場×コース×馬場状態のグループ内での斤量 vs タイム相関
df_nonan = df.dropna(subset=['finish_time_sec', 'weight_dev', 'weight_entry']).copy()

# 補正済みグループ内での偏差 (within-race normalization)
grp = ['race_id']
df_nonan['time_mean_race'] = df_nonan.groupby('race_id')['finish_time_sec'].transform('mean')
df_nonan['time_dev_race']  = df_nonan['finish_time_sec'] - df_nonan['time_mean_race']

# 単純 OLS: race fixed effects + weight_x_dist のみ
df_nonan['weight_x_dist'] = df_nonan['weight_dev'] * df_nonan['dist_scale']
df_nonan['fe_cell'] = (
    df_nonan['distance'].astype(str) + '_' +
    df_nonan.get('course', '').astype(str) + '_' +
    df_nonan['surface'] + '_' +
    df_nonan.get('track_condition', '').astype(str)
)

# レース内 OLS（固定効果なしで斤量偏差 → タイム偏差を回帰）
corr = df_nonan[['time_dev_race', 'weight_x_dist']].corr()
print(f'レース内タイム偏差 vs weight_x_dist の相関係数: {corr.iloc[0,1]:.4f}')

# 単純回帰
simple_model = smf.ols('time_dev_race ~ weight_x_dist', data=df_nonan).fit()
beta_simple = simple_model.params['weight_x_dist']
print(f'\n単純回帰 β₃（レース内偏差のみ）: {beta_simple:.4f} 秒/kg')
print(f'理論値 0.2 との差: {abs(beta_simple - 0.2):.4f}')

## 3. 距離帯別の感度分析

In [ ]:
def distance_band(d):
    if d < 1500:   return 'sprint (<1500m)'
    if d < 1800:   return 'mile (1500-1799m)'
    if d < 2400:   return 'middle (1800-2399m)'
    return 'long (>=2400m)'

df_nonan['distance_band'] = df_nonan['distance'].apply(distance_band)

results = []
for band, group in df_nonan.groupby('distance_band'):
    if len(group) < 100:
        continue
    try:
        m = smf.ols('time_dev_race ~ weight_x_dist', data=group).fit()
        b = m.params['weight_x_dist']
        ci = m.conf_int().loc['weight_x_dist']
        results.append({
            'band': band,
            'n': len(group),
            'beta3': b,
            'ci_lo': ci[0],
            'ci_hi': ci[1],
            'avg_dist': group['distance'].mean(),
            'theoretical': 0.2 * group['dist_scale'].mean(),
        })
    except Exception as e:
        print(f'{band}: 推定失敗 ({e})')

df_results = pd.DataFrame(results)
print('=== 距離帯別 β₃ ===')
print(df_results.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# 可視化: 推定値 vs 理論値
fig, ax = plt.subplots(figsize=(10, 5))

bands = df_results['band'].tolist()
x = np.arange(len(bands))

ax.bar(x - 0.2, df_results['beta3'], 0.4, label='推定値 β₃', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, df_results['theoretical'], 0.4, label='理論値 (0.2/kg×dist_scale)', color='coral', alpha=0.8)

for i, row in df_results.iterrows():
    ax.errorbar(x[i] - 0.2, row['beta3'],
                yerr=[[row['beta3']-row['ci_lo']], [row['ci_hi']-row['beta3']]],
                fmt='none', color='black', capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=15, ha='right')
ax.set_ylabel('秒/kg')
ax.set_title('距離帯別 β₃（斤量補正係数）: 推定値 vs 理論値')
ax.legend()
ax.axhline(0.2, color='red', linestyle=':', linewidth=1, label='理論値 0.2')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'weight_correction_by_band.png', dpi=120)
plt.show()

## 4. 牝馬オフセットの検証

In [ ]:
# 牡馬 vs 牝馬で同斤量ハンデ（牝馬 -2kg）が適切かを確認
df_mixed = df_nonan[df_nonan['sex'].isin(['牡', '牝'])].copy()

# 同条件のレースで牝馬と牡馬の実タイム差（斤量差が 2kg あることを考慮）
# 期待値: 牝馬は牡馬より 0.2 × 2 × dist_scale 秒遅い（仮に同等能力ならば）

sex_grp = df_mixed.groupby(['distance_band', 'sex'])['time_dev_race'].agg(['mean', 'std', 'count'])
print('=== 性別×距離帯のタイム偏差（レース内偏差の平均） ===')
print(sex_grp)
print()
print('牝馬 -2kg 補正の理論タイム差（2000m基準）:', 0.2 * 2.0, '秒')

## 5. 検証結論

In [ ]:
# 理論値との統計的一致検定
t_stat, p_val = stats.ttest_1samp(
    df_results['beta3'].values,
    0.2,  # 帰無仮説: β₃ = 0.2
)

print('=== 検証まとめ ===')
print(f'β₃ 全体（NB-02）: {beta_weight:.4f} 秒/kg')
print(f'β₃ 単純回帰（レース内偏差）: {beta_simple:.4f} 秒/kg')
print(f'距離帯別 β₃ の平均: {df_results["beta3"].mean():.4f} 秒/kg')
print(f'理論値 (0.2) との t検定: t={t_stat:.3f}, p={p_val:.4f}')
print()

if abs(beta_weight - 0.2) < 0.05:
    print('✅ β₃ は理論値 0.2秒/kg と整合的 → NB-02 の値をそのまま採用')
elif abs(beta_weight - 0.2) < 0.10:
    print('⚠️ β₃ は理論値から若干乖離 → データ量・馬場条件の偏りを確認')
else:
    print('❌ β₃ が理論値から大きく乖離 → 斤量データの品質・変数の定義を再確認')

# 結果を保存
df_results.to_parquet(OUTPUT_DIR / 'weight_validation.parquet', index=False)
print(f'\n検証結果を保存: {OUTPUT_DIR / "weight_validation.parquet"}')